<a href="https://colab.research.google.com/github/gunnsmart/skills/blob/main/%F0%9F%96%BC%EF%B8%8F_AI_Image_Upscaler_(HAT_%26_DAT_Model)_%E2%80%94_Fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖼️ AI Image Upscaler (HAT & DAT Model) — ขยายภาพคมชัดสูงด้วย AI

สมุดงานนี้รองรับการขยายภาพ Super-Resolution ด้วยสถาปัตยกรรม Transformer รุ่นใหม่ล่าสุด:
- **HAT (Hybrid Attention Transformer)**: โมเดลคมชัดสูงพิเศษ (Real_HAT_GAN_sharper, Real_HAT_GAN, HAT-L)
- **DAT (Dual Aggregation Transformer)**: โมเดลเก็บรายละเอียดระดับพรีเมียม

รองรับการเลือก Output:
- **4K Fast**: ลดเวลาเหลือเพียง **~45-60 วินาที/ภาพ** (ได้ไฟล์ 4K ~8-16 MP ผ่านเกณฑ์ Adobe Stock สบายๆ)
- **8K Native**: ขยายเต็มพิกัด 4x คมชัดสูงสุดระดับพรีเมียม (~3.5 นาที/ภาพ)

In [ ]:
#@title 📥 ขั้นตอนที่ 1: ติดตั้งเอนจินและดาวน์โหลดโมเดล (HAT / DAT) { display-mode: "form" }
import os, sys, subprocess

print("⏳ 1. กำลังเตรียมระบบและติดตั้งไลบรารีประมวลผล...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'spandrel', 'huggingface_hub', 'timm', 'einops', 'Pillow', 'opencv-python', '-q'])

from huggingface_hub import hf_hub_download

#@markdown ### 🤖 เลือกโมเดล AI ที่ต้องการใช้งาน:
โมเดล = "DAT_x4 (Dual Aggregation Transformer)" #@param ["Real_HAT_GAN_sharper (แนะนำ - ภาพจริงคมชัดสูงพิเศษ)", "Real_HAT_GAN_SRx4 (ภาพจริงเน้นความธรรมชาติ)", "HAT_SRx4_ImageNet-pretrain (โมเดลมาตรฐาน 4x)", "DAT_x4 (Dual Aggregation Transformer)"]

MODEL_CONFIG = {
    "Real_HAT_GAN_sharper": ("Acly/hat", "Real_HAT_GAN_sharper.pth", "Real_HAT_GAN_sharper.pth"),
    "Real_HAT_GAN_SRx4": ("Acly/hat", "Real_HAT_GAN_SRx4.pth", "Real_HAT_GAN_SRx4.pth"),
    "HAT_SRx4_ImageNet-pretrain": ("Acly/hat", "HAT_SRx4_ImageNet-pretrain.pth", "HAT_SRx4_ImageNet-pretrain.pth"),
    "DAT_x4": ("jaideepsingh/upscale_models", "DAT/DAT_x4.pth", "DAT_x4.pth")
}

key = "Real_HAT_GAN_sharper"
if "SRx4 (ภาพจริง" in โมเดล: key = "Real_HAT_GAN_SRx4"
elif "ImageNet" in โมเดล: key = "HAT_SRx4_ImageNet-pretrain"
elif "DAT" in โมเดล: key = "DAT_x4"

repo_id, file_in_repo, out_filename = MODEL_CONFIG[key]
os.makedirs('/content/weights', exist_ok=True)
dest_path = f'/content/weights/{out_filename}'

for old in os.listdir('/content/weights'):
    if old.endswith('.pth') and old != out_filename:
        try: os.remove(os.path.join('/content/weights', old))
        except: pass

print(f"📥 กำลังดาวน์โหลดโมเดล: {out_filename} จาก HuggingFace ({repo_id})...")
try:
    downloaded = hf_hub_download(
        repo_id=repo_id,
        filename=file_in_repo,
        local_dir='/content/weights'
    )
    actual_path = os.path.join('/content/weights', file_in_repo)
    if os.path.exists(actual_path) and actual_path != dest_path:
        os.rename(actual_path, dest_path)

    size_mb = os.path.getsize(dest_path) / (1024 * 1024)
    print(f"\n✅ ดาวน์โหลดโมเดลสำเร็จสมบูรณ์! ขนาด: {size_mb:.1f} MB")
    print("👉 เลื่อนลงไปกดปุ่ม Play ที่ขั้นตอนที่ 2 ด้านล่างได้เลย")
except Exception as e:
    print(f"\n❌ เกิดข้อผิดพลาดในการดาวน์โหลด: {e}")

⏳ 1. กำลังเตรียมระบบและติดตั้งไลบรารีประมวลผล...
📥 กำลังดาวน์โหลดโมเดล: DAT_x4.pth จาก HuggingFace (jaideepsingh/upscale_models)...


DAT/DAT_x4.pth: reconstructing file:   0%|          |  0.00B /  155MB            

DAT/DAT_x4.pth: downloading bytes:           |  0.00B            


✅ ดาวน์โหลดโมเดลสำเร็จสมบูรณ์! ขนาด: 147.5 MB
👉 เลื่อนลงไปกดปุ่ม Play ที่ขั้นตอนที่ 2 ด้านล่างได้เลย


In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files("Acly/hat")
print("ไฟล์ทั้งหมดใน Acly/hat:")
for f in files:
    print(f)

ไฟล์ทั้งหมดใน Acly/hat:
.gitattributes
HAT_SRx4_ImageNet-pretrain.pth
README.md
Real_HAT_GAN_sharper.pth


In [ ]:
#@title 🛠️ ขั้นตอนที่ 2: ตั้งค่า แล้วกด Play เพื่ออัปโหลดรูปภาพ { display-mode: "form" }
import os, sys, time, shutil, zipfile, math, gc
import cv2
import numpy as np
import torch
from PIL import Image, ImageEnhance
from google.colab import files
from spandrel import ModelLoader

#@markdown ---
#@markdown ### 🎯 1. ความละเอียด Output ที่ต้องการ:
ความละเอียด_Output = "8K Native (~3.5 นาที/ภาพ - คมชัดเต็มพิกัด 4x)" #@param ["4K Fast (~50 วินาที/ภาพ - แนะนำสำหรับภาพ 2K ส่ง Stock)", "8K Native (~3.5 นาที/ภาพ - คมชัดเต็มพิกัด 4x)", "4K Super-Sample (ขยาย 8K แล้วย่อเป็น 4K เพื่อความเนียนกริบ)"]

#@markdown ### 🧩 2. การจัดการขนาด Tile (ป้องกัน CUDA Out of Memory):
ขนาด_Tile = "512 (สำหรับ DAT)" #@param ["256 (แนะนำสูงสุดสำหรับ HAT ป้องกันแรมเต็ม 100%)", "192 (สำหรับภาพความละเอียดสูงมาก)", "128 (ประหยัดแรมสูงสุด)", "512 (สำหรับ DAT)"]

#@markdown ### 🔪 3. ปรับความคมชัดเพิ่มเติมหลังขยาย:
ปรับความคมชัด = "ปิด (ค่ามาตรฐานตามโมเดล)" #@param ["ปิด (ค่ามาตรฐานตามโมเดล)", "เล็กน้อย (10%)", "ปานกลาง (20%)", "สูง (30%)"]

#@markdown ### 📂 4. การบันทึกไฟล์:
นามสกุลไฟล์ = "JPEG" #@param ["JPEG", "PNG"]
คุณภาพ_JPEG = 100 #@param {type:"slider", min:80, max:100, step:1}

#@markdown ---

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    print("⚠️ คำเตือน: กำลังรันบน CPU ระบบจะทำงานช้า แนะนำให้เปิด GPU ที่ Runtime > Change runtime type > T4 GPU")

weights = [f for f in os.listdir('/content/weights') if f.endswith('.pth')] if os.path.exists('/content/weights') else []

if not weights:
    print("❌ ยังไม่พบไฟล์โมเดล กรุณาเลื่อนขึ้นไปกด Play ที่ขั้นตอนที่ 1 ให้เสร็จก่อน")
else:
    weight_path = os.path.join('/content/weights', weights[0])
    print(f"🚀 โหลดโมเดล: {weights[0]} บน {device} ...")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    descriptor = ModelLoader().load_from_file(weight_path)
    model = descriptor.model.to(device).eval()

    ext = '.jpg' if นามสกุลไฟล์ == 'JPEG' else '.png'
    initial_tile_size = 256
    if "192" in ขนาด_Tile: initial_tile_size = 192
    elif "128" in ขนาด_Tile: initial_tile_size = 128
    elif "512" in ขนาด_Tile: initial_tile_size = 512

    sharpen_amount = 0.0
    if "10%" in ปรับความคมชัด: sharpen_amount = 0.1
    elif "20%" in ปรับความคมชัด: sharpen_amount = 0.2
    elif "30%" in ปรับความคมชัด: sharpen_amount = 0.3

    is_4k_fast = "4K Fast" in ความละเอียด_Output
    is_4k_supersample = "4K Super-Sample" in ความละเอียด_Output

    def _execute_tile_upscale(img_tensor, model, tile=256, pad=16, scale=4):
        b, c, h, w = img_tensor.shape
        out_h, out_w = h * scale, w * scale
        output = torch.zeros((b, c, out_h, out_w), dtype=torch.float32, device=img_tensor.device)
        tiles_x = math.ceil(w / tile)
        tiles_y = math.ceil(h / tile)

        for y in range(tiles_y):
            for x in range(tiles_x):
                x_start = x * tile
                x_end = min(x_start + tile, w)
                y_start = y * tile
                y_end = min(y_start + tile, h)

                in_x_start = max(x_start - pad, 0)
                in_x_end = min(x_end + pad, w)
                in_y_start = max(y_start - pad, 0)
                in_y_end = min(y_end + pad, h)

                in_tile = img_tensor[:, :, in_y_start:in_y_end, in_x_start:in_x_end]
                out_tile = model(in_tile)

                out_x_start = (x_start - in_x_start) * scale
                out_x_end = out_x_start + (x_end - x_start) * scale
                out_y_start = (y_start - in_y_start) * scale
                out_y_end = out_y_start + (y_end - y_start) * scale

                output[:, :, y_start*scale:y_end*scale, x_start*scale:x_end*scale] = out_tile[:, :, out_y_start:out_y_end, out_x_start:out_x_end]
                del in_tile, out_tile
        return output

    def upscale_tensor_with_fallback(img_tensor, model, initial_tile=256, pad=16, scale=4):
        trial_tiles = [initial_tile]
        for t in [256, 192, 128, 96]:
            if t < initial_tile and t not in trial_tiles:
                trial_tiles.append(t)

        for cur_tile in trial_tiles:
            try:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                with torch.inference_mode():
                    return _execute_tile_upscale(img_tensor, model, tile=cur_tile, pad=pad, scale=scale)
            except torch.cuda.OutOfMemoryError:
                print(f"\n    ⚠️ VRAM เต็มที่ tile={cur_tile} → ปรับลด tile อัตโนมัติ...", end="", flush=True)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    gc.collect()
            except Exception as e:
                raise e

        print(f"\n    ⚠️ สลับไปประมวลผลบน CPU ชั่วคราว...", end="", flush=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model_cpu = model.to('cpu')
        img_cpu = img_tensor.to('cpu')
        with torch.inference_mode():
            res = _execute_tile_upscale(img_cpu, model_cpu, tile=256, pad=pad, scale=scale)
        model.to(device)
        return res

    OUTPUT_DIR = '/content/upscaled_output'
    if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("\n📤 เลือกรูปภาพจากเครื่องของคุณ (เลือกพร้อมกันได้หลายรูป):")
    uploaded = files.upload()

    if not uploaded:
        print("❌ ไม่ได้เลือกรูปภาพ กรุณากด Play ใหม่อีกครั้ง")
    else:
        total_imgs = len(uploaded)
        mode_label = "4K Fast" if is_4k_fast else ("4K Super-Sample" if is_4k_supersample else "8K Native")
        print(f"\n🚀 เริ่มประมวลผลโหมด [{mode_label}] ({total_imgs} ภาพ)...")
        start_all = time.time()
        success_count = 0

        for idx, (filename, file_bytes) in enumerate(uploaded.items()):
            print(f"🖼️ [{idx+1}/{total_imgs}] กำลังขยายภาพ: {filename} ...", end="", flush=True)

            nparr = np.frombuffer(file_bytes, np.uint8)
            img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            if img_bgr is None:
                print(" ⚠️ อ่านไฟล์ไม่ได้ ข้าม")
                continue

            orig_h, orig_w = img_bgr.shape[:2]

            # กรณีเลือก 4K Fast: ย่อขนาดต้นฉบับลงครึ่งหนึ่งก่อนเข้า HAT 4x
            # (เช่น 2048x1152 -> 1024x576 -> HAT 4x -> 4096x2304 ระดับ 4K ในเวลา ~50 วินาที)
            if is_4k_fast and (orig_w > 1920 or orig_h > 1080):
                target_in_w = orig_w // 2
                target_in_h = orig_h // 2
                img_bgr_proc = cv2.resize(img_bgr, (target_in_w, target_in_h), interpolation=cv2.INTER_AREA)
            else:
                img_bgr_proc = img_bgr

            img_rgb = cv2.cvtColor(img_bgr_proc, cv2.COLOR_BGR2RGB)
            img_t = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float().div(255.0).unsqueeze(0).to(device)

            output_t = upscale_tensor_with_fallback(img_t, model, initial_tile=initial_tile_size, pad=16, scale=4)

            output_np = output_t.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
            output_img = (output_np * 255.0).round().astype(np.uint8)
            del img_t, output_t

            pil_img = Image.fromarray(output_img)

            # กรณีเลือก 4K Super-Sample: ย่อจาก 8K ลงมาเป็น 4K (กว้าง x2 จากต้นฉบับ)
            if is_4k_supersample and (orig_w > 1920 or orig_h > 1080):
                target_4k_w = orig_w * 2
                target_4k_h = orig_h * 2
                pil_img = pil_img.resize((target_4k_w, target_4k_h), Image.LANCZOS)

            final_w, final_h = pil_img.size

            if sharpen_amount > 0:
                enhancer = ImageEnhance.Sharpness(pil_img)
                pil_img = enhancer.enhance(1.0 + sharpen_amount)

            stem = os.path.splitext(filename)[0]
            tag = "4K" if (is_4k_fast or is_4k_supersample) else "8K"
            save_path = os.path.join(OUTPUT_DIR, f"{stem}_{tag}{ext}")

            if ext == '.jpg':
                pil_img.save(save_path, 'JPEG', quality=คุณภาพ_JPEG, dpi=(300, 300), optimize=True)
            else:
                pil_img.save(save_path, 'PNG', dpi=(300, 300))

            mp = (final_w * final_h) / 1e6
            print(f" ✅ เสร็จ ({orig_w}x{orig_h} → {final_w}x{final_h} | {mp:.1f}MP)")
            success_count += 1

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        total_time = time.time() - start_all
        print(f"\n⏱️ รวมเวลาประมวลผลทั้งหมด: {total_time:.1f} วินาที | สำเร็จ {success_count}/{total_imgs} ภาพ")

        if success_count > 0:
            zip_path = "/content/AI_Upscaled_Images.zip"
            if os.path.exists(zip_path): os.remove(zip_path)
            with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for f in os.listdir(OUTPUT_DIR):
                    zipf.write(os.path.join(OUTPUT_DIR, f), f)

            print("📦 กำลังส่งไฟล์ ZIP ลงเครื่องให้อัตโนมัติ...")
            files.download(zip_path)

🚀 โหลดโมเดล: DAT_x4.pth บน cuda ...


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



📤 เลือกรูปภาพจากเครื่องของคุณ (เลือกพร้อมกันได้หลายรูป):


Saving 1058991134_1790741551822130.jpg to 1058991134_1790741551822130.jpg
Saving 19830623_1790741548246918.jpg to 19830623_1790741548246918.jpg
Saving 588015866_1790741535120844.jpg to 588015866_1790741535120844.jpg
Saving 327361539_1790741543344345.jpg to 327361539_1790741543344345.jpg

🚀 เริ่มประมวลผลโหมด [8K Native] (4 ภาพ)...
🖼️ [1/4] กำลังขยายภาพ: 1058991134_1790741551822130.jpg ... ✅ เสร็จ (1376x768 → 5504x3072 | 16.9MP)
🖼️ [2/4] กำลังขยายภาพ: 19830623_1790741548246918.jpg ... ✅ เสร็จ (1200x896 → 4800x3584 | 17.2MP)
🖼️ [3/4] กำลังขยายภาพ: 588015866_1790741535120844.jpg ... ✅ เสร็จ (1200x896 → 4800x3584 | 17.2MP)
🖼️ [4/4] กำลังขยายภาพ: 327361539_1790741543344345.jpg ... ✅ เสร็จ (1376x768 → 5504x3072 | 16.9MP)

⏱️ รวมเวลาประมวลผลทั้งหมด: 89.5 วินาที | สำเร็จ 4/4 ภาพ
📦 กำลังส่งไฟล์ ZIP ลงเครื่องให้อัตโนมัติ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>